In [146]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/valakhorasani/gym-members-exercise-dataset/gym_members_exercise_tracking.csv


In [147]:
df = pd.read_csv('/kaggle/input/datasets/valakhorasani/gym-members-exercise-dataset/gym_members_exercise_tracking.csv')
df.head()

,Age,Gender,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Workout_Type,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI
0,56,Male,88.3,1.71,180,157,60,1.69,1313.0,Yoga,12.6,3.5,4,3,30.20
1,46,Female,74.9,1.53,179,151,66,1.30,883.0,HIIT,33.9,2.1,4,2,32.00
2,32,Female,68.1,1.66,167,122,54,1.11,677.0,Cardio,33.4,2.3,4,2,24.71
3,25,Male,53.2,1.70,190,164,56,0.59,532.0,Strength,28.8,2.1,3,1,18.41
4,38,Male,46.1,1.79,188,158,68,0.64,556.0,Strength,29.2,2.8,3,1,14.39


In [148]:
print(df.info())
print(df.isna().sum())
print(df.Workout_Type.unique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 973 entries, 0 to 972
Data columns (total 15 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   Age                            973 non-null    int64  
 1   Gender                         973 non-null    object 
 2   Weight (kg)                    973 non-null    float64
 3   Height (m)                     973 non-null    float64
 4   Max_BPM                        973 non-null    int64  
 5   Avg_BPM                        973 non-null    int64  
 6   Resting_BPM                    973 non-null    int64  
 7   Session_Duration (hours)       973 non-null    float64
 8   Calories_Burned                973 non-null    float64
 9   Workout_Type                   973 non-null    object 
 10  Fat_Percentage                 973 non-null    float64
 11  Water_Intake (liters)          973 non-null    float64
 12  Workout_Frequency (days/week)  973 non-null    int

In [149]:

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import OneHotEncoder

class CustomEncoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.encoder = OneHotEncoder(
            sparse_output=False,          # dense array (easier to work with DataFrames)
            handle_unknown="ignore",      # safer for unseen categories
            drop=None                     # keep all categories (or use "first" if you prefer)
        )
        self.feature_names_ = None

    def fit(self, X, y=None):
        # Expect a DataFrame with the two categorical columns
        self.encoder.fit(X[["Gender", "Workout_Type"]])
        self.feature_names_ = self.encoder.get_feature_names_out(
            ["Gender", "Workout_Type"]
        )
        return self

    def transform(self, X):
        X = X.copy()

        # One-hot encode the two columns
        encoded = self.encoder.transform(X[["Gender", "Workout_Type"]])
        encoded_df = pd.DataFrame(
            encoded,
            columns=self.feature_names_,
            index=X.index
        )

        # Drop original categorical columns and join the encoded ones
        X = X.drop(columns=["Gender", "Workout_Type"])
        X = X.join(encoded_df)

        return X

In [150]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline ([
    ('customencoder',CustomEncoder())
])

In [151]:
df = pipeline.fit_transform(df)

In [152]:
df.head()

,Age,Weight (kg),Height (m),Max_BPM,Avg_BPM,Resting_BPM,Session_Duration (hours),Calories_Burned,Fat_Percentage,Water_Intake (liters),Workout_Frequency (days/week),Experience_Level,BMI,Gender_Female,Gender_Male,Workout_Type_Cardio,Workout_Type_HIIT,Workout_Type_Strength,Workout_Type_Yoga
0,56,88.3,1.71,180,157,60,1.69,1313.0,12.6,3.5,4,3,30.20,0.0,1.0,0.0,0.0,0.0,1.0
1,46,74.9,1.53,179,151,66,1.30,883.0,33.9,2.1,4,2,32.00,1.0,0.0,0.0,1.0,0.0,0.0
2,32,68.1,1.66,167,122,54,1.11,677.0,33.4,2.3,4,2,24.71,1.0,0.0,1.0,0.0,0.0,0.0
3,25,53.2,1.70,190,164,56,0.59,532.0,28.8,2.1,3,1,18.41,0.0,1.0,0.0,0.0,1.0,0.0
4,38,46.1,1.79,188,158,68,0.64,556.0,29.2,2.8,3,1,14.39,0.0,1.0,0.0,0.0,1.0,0.0


In [153]:
from sklearn.preprocessing import StandardScaler
X = df.drop("Calories_Burned", axis=1)
y = df["Calories_Burned"]

scaler = StandardScaler()
X_data = scaler.fit_transform(X) #use our standard scaler
y_data = y.to_numpy() #turn it into numpy array

In [154]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV

# 1. Define the base model
reg = RandomForestRegressor(random_state=42)   # or whatever other fixed params you want

# 2. Define the parameter grid for the RandomForest
param_grid = {
    'n_estimators': [50, 100],
    'max_depth': [15, 25],
    'max_features': ['sqrt'],
    'min_samples_split': [5],
    'min_samples_leaf': [2]
}

# 3. Create the GridSearchCV on the base model
grid_search = GridSearchCV(
    estimator=reg,
    param_grid=param_grid,
    cv=3,                 # you reduced it to 3 – fine
    scoring='r2',
    n_jobs=-1,
    verbose=1
)

# 4. Fit
grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 4 candidates, totalling 12 fits


GridSearchCV(cv=3, estimator=RandomForestRegressor(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [15, 25], 'max_features': ['sqrt'],
                         'min_samples_leaf': [2], 'min_samples_split': [5],
                         'n_estimators': [50, 100]},
             scoring='r2', verbose=1)

In [155]:
best_params = grid_search.best_params_

In [156]:

from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_dist = {
    'n_estimators': randint(100, 300),
    'max_depth': [None, 10, 20, 30, 40, 50],
    'min_samples_split': randint(2, 11),
    'min_samples_leaf': randint(1, 11),
    'max_features': ['sqrt', None]
}

reg = RandomForestRegressor(n_jobs=-1)
random_search = RandomizedSearchCV(reg, param_distributions=param_dist, n_iter=2, cv=3, scoring='neg_root_mean_squared_error', verbose=2, random_state =10, n_jobs=-1)
random_search.fit(X_train, y_train)
     


Fitting 3 folds for each of 2 candidates, totalling 6 fits


RandomizedSearchCV(cv=3, estimator=RandomForestRegressor(n_jobs=-1), n_iter=2,
                   n_jobs=-1,
                   param_distributions={'max_depth': [None, 10, 20, 30, 40, 50],
                                        'max_features': ['sqrt', None],
                                        'min_samples_leaf': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c2d7f0b4890>,
                                        'min_samples_split': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c2d802dd9d0>,
                                        'n_estimators': <scipy.stats._distn_infrastructure.rv_discrete_frozen object at 0x7c2d802dd1c0>},
                   random_state=10, scoring='neg_root_mean_squared_error',
                   verbose=2)

In [157]:

best_regressor = random_search.best_estimator_
best_regressor.score(X_test, y_test)

0.9691660043369

In [158]:
import math
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = best_regressor.predict(X_test)
print("R2:", r2_score(y_test, y_pred))
print("MSE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred))

R2: 0.9691660043369001
MSE: 34.048608156488505
RMSE: 2358.4767057127087


In [159]:
best_regressor = random_search.best_estimator_
best_regressor.score(X_test, y_test)

0.9691660043369001

In [160]:


import math
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

y_pred = best_regressor.predict(X_test)
print("R2:", r2_score(y_test, y_pred))
print("MSE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", mean_squared_error(y_test, y_pred))
     


R2: 0.9691660043369001
MSE: 34.04860815648851
RMSE: 2358.4767057127096
